# ENGR 1451/2451 — Weekly Assignment 10
**10 points**

This assignment focuses on three goals:
(i) interpreting Type I and Type II errors, sensitivity, and positive predictive value using **real biopsy-image data**;
(ii) investigating how significance cutoffs and sample size affect power, using measured features from the same dataset; and
(iii) comparing independent groups and paired observations with $t$-tests.


**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE10-1. Real data: diagnostic flags and base rates (3 pts)

**Dataset:** `./data/LE10_wisconsin_tumors.csv` contains **569 real, labeled biopsy-image observations** from the Wisconsin Diagnostic Breast Cancer dataset. Each row represents a breast mass, with a known diagnosis (`benign` or `malignant`) and measurements extracted from images of cell nuclei. You will use the feature `mean_radius`.

For this classroom exercise, suppose a simple imaging rule **flags** a case when `mean_radius >= 14.0`. A flag means **the rule predicts malignant**; a non-flag means it predicts benign. The threshold is chosen for teaching, **not** as a validated medical screening rule.

The data come from a selected group of diagnostic cases, **not a random population-screening sample**. We will therefore distinguish (1) rates calculated from these real observations from (2) projections under a *hypothetical* population prevalence.

**Source:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic), Wolberg, Mangasarian, Street & Street (1993), DOI: 10.24432/C5DW2B (CC BY 4.0). This teaching CSV contains original real observations but only two of the measured features.

### A. Build the screening table and examine prevalence (2 pts)

Using Python:

1. Load the CSV with `pd.read_csv()` and show the number of cases in each diagnosis category.
2. Using the fixed threshold above, count **true positives (TP)**, **false negatives (FN)**, **false positives (FP)**, and **true negatives (TN)**. Treat *malignant* as the positive class.
3. Display a $2\times 2$ `pd.DataFrame` with **Flagged / Not flagged** as rows and **Malignant / Benign** as columns.
4. Calculate the observed **sensitivity**, **specificity**, **positive predictive value (PPV)**, and **accuracy**:

$$
\mathrm{Sensitivity}=\frac{TP}{TP+FN},\quad
\mathrm{Specificity}=\frac{TN}{TN+FP},\quad
\mathrm{PPV}=\frac{TP}{TP+FP},\quad
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}.
$$

5. **Base-rate thought experiment:** Suppose this rule is applied to **200,000** future cases in a hypothetical population where **0.5%** are malignant. Assume the sensitivity and specificity found in this CSV would transfer unchanged. Compute the *expected* TP, FN, FP, and TN, then the projected PPV. Expected counts need not be whole numbers; round only when displaying them. Compare the projected PPV with the PPV from the observed dataset.

Report percentages to **one decimal place**.


In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 4)

cases = pd.read_csv("./data/wisconsin_tumors.csv")
display(cases.head())
print(cases["diagnosis"].value_counts())

threshold = 14.0
# A case is flagged when mean_radius >= threshold.
# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.
# Compute empirical sensitivity, specificity, PPV, and accuracy.
# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.
# Compute the projected PPV (using unrounded expected counts).

# Your code here
TP = len(cases[(cases["diagnosis"] == "malignant")&(cases["mean_radius"] >= threshold)])
TN = len(cases[(cases["diagnosis"] == "benign")&(cases["mean_radius"] < threshold)])
FN = len(cases[(cases["diagnosis"] == "malignant")&(cases["mean_radius"] < threshold)])
FP = len(cases[(cases["diagnosis"] == "benign")&(cases["mean_radius"] >= threshold)])

table = pd.DataFrame(
    {
        "Malignant": [TP, FN],
        "Benign": [FP, TN]
    },
    index=["Flagged","Not flagged"]
)

sensitivity = TP/(TP+FN)
specificity = TN/(TN+FP)
ppv = TP/(TP+FP)
accuracy = (TP+TN)/(TP+TN+FP+FN)

# Projection
mal_rate = 0.005 
population = 200000
true_mal = population*mal_rate
true_ben = population-true_mal

TP_proj = sensitivity*true_mal
FN_proj = true_mal - sensitivity*true_mal
FP_proj = true_ben - specificity*true_ben
TN_proj = specificity*true_ben

print(f"TP: {TP_proj:.0f}")
print(f"TN: {TN_proj:.0f}")
print(f"FP: {FP_proj:.0f}")
print(f"FN: {FN_proj:.0f}")

ppv_proj = TP_proj/(TP_proj+FP_proj)

print(f"PPV projected: {ppv_proj*100:.1f}%")
print(f"PPV observed: {ppv*100:.1f}%")

table.head()

,diagnosis,mean_radius,mean_smoothness
0,malignant,17.99,0.1184
1,malignant,20.57,0.0847
2,malignant,19.69,0.1096
3,malignant,11.42,0.1425
4,malignant,20.29,0.1003


diagnosis
benign       357
malignant    212
Name: count, dtype: int64
TP: 849
TN: 170014
FP: 28986
FN: 151
PPV projected: 2.8%
PPV observed: 77.6%


,Malignant,Benign
Flagged,180,52
Not flagged,32,305


### B. Interpret the results (1 pt)

Answer in about **3–4 sentences**:

1. For this screening rule, what is a **Type I error** (false positive)? What is a **Type II error** (false negative)?
2. Why does the PPV change so much when the assumed prevalence changes, even if sensitivity and specificity stay fixed? Does a specificity near 85% imply that 85% of flagged cases are malignant?
3. Why should you **not** claim that the observed proportion of malignant cases in this dataset is the prevalence in the general population?

**Answer:**

A Type One error would be benign tumor that gets flagged, and a type Two error would be a malignant tumor that does not get flagged. The PPV changes so much because the percentage of actual malignant tumors changes, and the test does not. The test flags the same percentage of tumors, but the fractional amount of tumors goes down with the projection. The datasets fraction cannot be projected onto a population due to the fact that not everyone in a population undergoes this testing. People who are reccomended for screening are generally the ones who undergo the testing, and there are generally other factors that would cause someone to undergo certain testing.


## LE10-2. Significance cutoffs, $\beta$, and power (3 pts)

Use the **same real CSV**, but now study another imaging feature, `mean_smoothness`. We are designing a **population-level test based on the mean of $n$ independent biopsies**, *not* screening an individual case. Unlike LE10-1, which uses a fixed threshold to classify individual cases, we now choose cutoffs to control the probability of a Type I error at a specified significance level, $\alpha$.

Estimate the mean and sample SD of `mean_smoothness` separately for **benign** and **malignant** cases in the CSV. For this *planning exercise*, treat those estimated values as fixed benchmarks. Under a **normal approximation for sample means**, model

$$
\bar X\mid H_0 \approx N\!\left(\mu_0,\frac{\sigma_0^2}{n}\right),\qquad
\bar X\mid H_A \approx N\!\left(\mu_1,\frac{\sigma_1^2}{n}\right),
$$

where $H_0$ uses the **benign** reference distribution and the specified $H_A$ uses the **malignant** distribution. A false alarm means rejecting the benign reference when it is true; a miss means failing to reject it when the malignant model is true. **These are approximate design calculations**, not an evaluation of a clinical test or a claim that all biopsy measurements are normally distributed.

Use a **two-sided** decision rule: flag a sample if its mean is below the lower cutoff **or** above the upper cutoff. Compute cutoffs from the *benign* sampling distribution; compute $\beta$ as the probability of falling **between those cutoffs under the malignant model**.

| Plan | Independent biopsies $n$ | Significance level $\alpha$ |
|---|---:|---:|
| A | 10 | 0.05 |
| B | 10 | 0.01 |
| C | 25 | 0.01 |

### A. Calculate error rates and power (2 pts)

Using Python:

1. Extract `mean_smoothness` for benign and malignant cases; calculate each group's sample mean and sample SD using `.std(ddof=1)`.
2. For each plan, define `benign_model` and `malignant_model` with `stats.norm(loc=..., scale=SD/np.sqrt(n))`.
3. Calculate the **lower and upper cutoffs** using `benign_model.ppf([alpha/2, 1-alpha/2])`.
4. Compute $\beta= P(\text{not flagged}\mid H_A)$ using `malignant_model.cdf(upper) - malignant_model.cdf(lower)` and **power** $=1-\beta$.
5. Display a table with plan, $n$, $\alpha$, both cutoffs, $\beta$, and power. Report probabilities to **three decimal places**.

*Use the actual measurements to estimate model parameters; do not simulate additional biopsy records.*


In [2]:
# Use the dataset loaded in LE10-1.
benign = cases.loc[cases["diagnosis"] == "benign", "mean_smoothness"]
malignant = cases.loc[cases["diagnosis"] == "malignant", "mean_smoothness"]
pd.set_option("display.precision", 3)

# Calculate mu0, sd0 (benign), and mu1, sd1 (malignant) from the CSV.
# Print both pairs of estimates.

m0, m1 = benign.mean(), malignant.mean()
sd0, sd1 = benign.std(ddof=1), malignant.std(ddof=1)

print(f"mu0: {m0:.3f}, mu1 {m1:.3f}")
print(f"sd0: {sd0:.3f}, sd1 {sd1:.3f}")

plans = pd.DataFrame({
    "plan": ["A", "B", "C"],
    "n": [10, 10, 25],
    "alpha": [0.05, 0.01, 0.01],
})

rows = []

for i in range(0,3):
    plan = plans["plan"][i]
    n = plans["n"][i]
    alpha = plans["alpha"][i]
    benign_model = stats.norm(loc=m0, scale=sd0/np.sqrt(n))
    malignant_model = stats.norm(loc=m1, scale=sd1/np.sqrt(n))
    lower, upper = benign_model.ppf([alpha/2, 1-alpha/2])
    beta = malignant_model.cdf(upper) - malignant_model.cdf(lower)
    power = 1 - beta
    rows.append({"Plan": plan, "upper_cutoff": upper, "lower_cutoff": lower, "beta": beta, "power": power, "alpha": alpha, "n": n})

summary = pd.DataFrame(rows)

display(summary)

# Your code here

mu0: 0.092, mu1 0.103
sd0: 0.013, sd1 0.013


,Plan,upper_cutoff,lower_cutoff,beta,power,alpha,n
0,A,0.101,0.084,0.300,0.700,0.05,10
1,B,0.103,0.082,0.553,0.447,0.01,10
2,C,0.099,0.086,0.083,0.917,0.01,25


### B. Explain the tradeoffs (1 pt)

Answer in about **2–3 sentences**:

1. Compare plans **A and B**: What happens to false alarms ($\alpha$) and missed detections ($\beta$) when the cutoff becomes stricter but $n$ stays the same?
2. Compare plans **B and C**: How does using more **independent biopsies** increase power at the same $\alpha$? Why would this be a study-design consideration rather than a way to improve diagnosis of a *single* patient?

**Answer:**
Missed detections go up and false alarms go up when cutoff becomes stricter. Additional independent observations get you much closer to a population than repeated or a small amount of biopsies.

## LE10-3. Comparing two mean temperatures (4 pts)

Choose the analysis based on the **study design**, as in w10b. Both parts concern peak battery-module temperature during a standardized load test; **lower is better**.

*The datasets in this problem are simulated classroom examples, not measurements from a published study.*

### A. Independent modules: new vs. standard cooling (2.5 pts)

The file `./data/LE10_battery_cooling.csv` contains peak temperatures ($^\circ$C) for **26 different battery modules**: 12 were randomly assigned to **new** cooling and 14 to **standard** cooling. Each module was measured once.

Let $\Delta=\mu_{\mathrm{new}}-\mu_{\mathrm{standard}}$. We want to know whether the new method **reduces** the population mean peak temperature. Test the boundary null $H_0:\Delta=0$ against $H_A:\Delta<0$, using $\alpha=0.05$.

Assume observations from different modules are independent, group distributions are approximately normal without strong outliers, and population SDs are unknown. Use **Welch's** two-sample $t$-test (do not assume equal population variances).

In [3]:
cooling = pd.read_csv("./data/battery_cooling.csv")

# The order is new minus standard: a negative difference favors the new method.
new = cooling.loc[cooling["method"] == "new", "peak_temperature_C"].to_numpy()
standard = cooling.loc[cooling["method"] == "standard", "peak_temperature_C"].to_numpy()

display(cooling.head())
print(cooling["method"].value_counts())

,module_id,method,peak_temperature_C
0,N01,new,60.1
1,N02,new,63.0
2,N03,new,59.4
3,N04,new,62.2
4,N05,new,57.8


method
standard    14
new         12
Name: count, dtype: int64


Use Python to:

1. Calculate each group's **sample size, mean, and sample SD** (with `ddof=1`), and the observed difference $\bar x_{\mathrm{new}}-\bar x_{\mathrm{standard}}$.
2. Calculate the estimated SE of the difference:

$$
\widehat{SE}=\sqrt{\frac{s_{\mathrm{new}}^2}{n_{\mathrm{new}}}+\frac{s_{\mathrm{standard}}^2}{n_{\mathrm{standard}}}}.
$$

3. Run `stats.ttest_ind(new, standard, equal_var=False, alternative="less")`. Print the **$t$ statistic, one-sided $P$-value**, and decision at $\alpha=0.05$.
4. For context, run the same Welch test with `alternative="two-sided"` and use its `.confidence_interval(confidence_level=0.95)` method to report a **two-sided 95% confidence interval** for $\Delta$. (You do not need to calculate Welch degrees of freedom manually.)

**Answer:**

In [4]:
# Calculate group summaries (n, sample mean, sample SD).
# Calculate the observed difference and estimated SE.
# Run Welch's one-sided test and print the decision at alpha = 0.05.
# Run a two-sided Welch test and print its 95% confidence interval.

# Your code here
alpha = 0.05

n_new, n_stan = len(new), len(standard)
m_new, m_stan = new.mean(), standard.mean()
sd_new, sd_stan = new.std(ddof=1), standard.std(ddof=1)

observed_diff = m_new - m_stan
estimated_se = np.sqrt((sd_new**2/n_new)+(sd_stan**2/n_stan))

one_sided = stats.ttest_ind(new, standard, equal_var=False, alternative="less")
two_sided = stats.ttest_ind(new, standard, equal_var=False, alternative="two-sided")

print(f"T statistic: {one_sided.statistic:.3f}, P value: {one_sided.pvalue}")
print(f"New process statistically different from standard? {one_sided.pvalue < alpha}")
print(f"Two sided 95% interval: {two_sided.confidence_interval(confidence_level=0.95)}")

T statistic: -4.386, P value: 9.944447668138834e-05
New process statistically different from standard? True
Two sided 95% interval: ConfidenceInterval(low=-5.390566889826646, high=-1.940385491125737)


### B. Matched modules: before vs. after a firmware change (1.5 pts)

A **separate experiment** measures the *same eight modules* before and after a cooling-control firmware update. The arrays are aligned: `before[i]` and `after[i]` belong to the same module.

1. Calculate the eight within-module differences `after - before` and their mean.
2. Run `stats.ttest_rel(after, before, alternative="less")` to test $H_0:\mu_d=0$ against $H_A:\mu_d<0$, where $d=\text{after}-\text{before}$. Print $t$, the one-sided $P$-value, and the decision at $\alpha=0.05$.
3. In **2–3 sentences total**, interpret the conclusions in parts A and B. Explain **why a paired rather than independent test** is required in part B and whether a statistically significant result by itself establishes the engineering importance of a temperature change.

**Answer:**

A paired test is required in Part B because the same modules were run twice. This means that there is a direct relation between before and after results, and may suggest that changes could be more statistically relevant due to the fact that the same modules were used. Part A and Part B both showed statistical significance in the difference between the new and standard cooling systems. The data in part B is more convincing as the modules in Part A could have simply been different without the cooling systems, but in part B a noticable difference on the same modules is present from before to after.

In [5]:
# Same modules measured twice; each index identifies the same module.
before = np.array([72, 68, 71, 70, 74, 69, 73, 67])
after  = np.array([69, 65, 70, 67, 73, 66, 72, 66])

# Calculate and display within-module differences and their mean.
# Run the paired t-test, report t and P, and make the decision at alpha = 0.05.

differences = after-before
mean = differences.mean()

print(f"Within-module differences: {differences}, mean: {mean}")

one_sided_test = stats.ttest_rel(after, before, alternative="less")
print(f"T-statistic: {one_sided_test.statistic}, p-value: {one_sided_test.pvalue}")
print(f"New process statistically different from standard? {one_sided_test.pvalue < alpha}")

Within-module differences: [-3 -3 -1 -3 -1 -3 -1 -1], mean: -2.0
T-statistic: -5.291502622129181, p-value: 0.0005668915516798226
New process statistically different from standard? True


## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Session records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*

{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA10",
  "export_n": 1,
  "session_date": "2026-11-10",
  "messages": [
    {"i": 1, "who": "student", "text": "whats the best way to construct my screening table, i have done TP TN FP FN and then the sensitivity values etc but im strugglign to put my TP TN FP FN into a nice table"},
    {"i": 2, "who": "tutor", "text": "Nice work getting all four counts and the rates done. The table is just a matter of telling `pd.DataFrame` where each number goes. [trimmed: generic leak-sensor example of pd.DataFrame with a dict of columns and index=, notes on column order, the required index, a column-sum sanity check, and pd.crosstab relabeling] For your table, the rows are Flagged / Not flagged and the columns are Malignant / Benign. Which two of your four counts go in the Malignant column, and in what order? If you build it and want a check, paste the cell and its output."},
    {"i": 3, "who": "student", "text": "[image uploaded: 2x2 table with rows Flagged / Not Flagged and columns Malignant / Benign; Flagged 180, 32; Not Flagged 52, 305]", "code": true},
    {"i": 4, "who": "tutor", "text": "The table looks clean, and the total checks out at 569. One thing needs fixing: two of the counts are in each other's cells. [trimmed: column sums 232 and 337 vs printed 212 malignant and 357 benign; off-diagonal 32 and 52 swapped; check whether the FN and FP variables or only the positions are swapped] One small thing: the prompt's row label is Not flagged, with a lowercase f."},
    {"i": 5, "who": "student", "text": "i needed to swap FP and FN"},
    {"i": 6, "who": "tutor", "text": "That's right: with those two swapped back, both columns match the printed diagnosis counts. If your `FP` and `FN` variables were the swapped ones (not just their positions in the table), rerun the cell so the rates are recalculated from the corrected values. Then paste the sensitivity, specificity, PPV, and accuracy if you'd like them checked. Otherwise, the next piece is the 200,000-case projection. Which would you like to do?"},
    {"i": 7, "who": "student", "text": "just tabel positions"},
    {"i": 8, "who": "tutor", "text": "Good, then your rates were built from the right variables and don't need recalculating. Want to move on to the projection, or paste the rates for a quick check first?"},
    {"i": 9, "who": "student", "text": "yes the projection"},
    {"i": 10, "who": "tutor", "text": "Nice, the table is done. The projection works in two stages. [trimmed: split 200,000 by diagnosis at 0.005; sensitivity splits the malignant count into TP and FN; specificity splits the benign count into TN and FP via 1 - specificity; keep unrounded] To start: how many of the 200,000 cases would you expect to be malignant, and how many benign?"},
    {"i": 11, "who": "student", "text": "[image uploaded: TP: 776, TN: 224, FP: 18896, FN: 180104, PPV projected: 0.03943990907885124] this is my projected data, how does it look", "code": true},
    {"i": 12, "who": "tutor", "text": "Good progress: the counts add to 200,000 and FP correctly uses 1 − specificity. Two things are off, though, and the sum checks will show you both. [trimmed: TN and FN labels swapped; sensitivity of 77.6% and specificity of about 90.5% imply FN and FP were swapped in the variables; print sensitivity and specificity; show projected PPV as a one-decimal percentage beside observed PPV] What do `sensitivity` and `specificity` print right now?"},
    {"i": 13, "who": "student", "text": "```\n0.7758620689655172 for sens\n```", "code": true},
    {"i": 14, "who": "tutor", "text": "That confirms it: 0.776 is 180/(180 + 52), so your `FN` variable currently holds the false-positive count. [trimmed: check the masks for FN (malignant and not flagged) and FP (benign and flagged); rebuild the table from the variables; rerun] Rerun the cell and paste the new rates and projected counts, and I'll check them."},
    {"i": 15, "who": "student", "text": "```\n0.8490566037735849 now this is sens\n```", "code": true},
    {"i": 16, "who": "tutor", "text": "That's correct: 180/212 is about 84.9%. The mask fix worked. Now paste the rest of the rerun output, meaning specificity, PPV, accuracy, and the projected counts and PPV, and I'll check them against the data."},
    {"i": 17, "who": "student", "text": "[image uploaded: TP: 849, TN: 151, FP: 28986, FN: 170014, PPV projected: 2.8%] for the projected", "code": true},
    {"i": 18, "who": "tutor", "text": "These numbers now match my reference calculation. PPV, TP, and FP are all correct, and the rounding is fine for display. The labels on the TN and FN lines are still swapped, though. [trimmed: sum checks TP + FN = 1,000 and FP + TN = 199,000] After that, the last piece of step 5 is to print your observed PPV beside the projected 2.8% so the comparison appears in the output. What does your observed PPV come to?"},
    {"i": 19, "who": "student", "text": "[image uploaded: TP: 849, TN: 170014, FP: 28986, FN: 151, PPV projected: 2.8%]", "code": true},
    {"i": 20, "who": "tutor", "text": "That's right: all four expected counts now sit under the correct labels, and both sum checks hold. To finish LE10-1A, print your observed PPV next to the projected 2.8% so the comparison shows in the output, and make sure your four observed rates are displayed as percentages to one decimal place. Once that's in, LE10-1A is complete. Want to keep going to LE10-1B?"},
    {"i": 21, "who": "student", "text": "[image uploaded: PPV projected: 2.8%, PPV observed: 77.59%]", "code": true},
    {"i": 22, "who": "tutor", "text": "The value is correct: 180/232 is about 77.6%. The prompt asks for percentages to one decimal place, though, so change that format to `:.1%`. It should then print 77.6%, matching the style of your projected line. With that change, LE10-1A is done: counts, table, rates, projection, and the comparison. Do you want to start LE10-1B, the three written questions about these results?"},
    {"i": 23, "who": "student", "text": "i could use some help on 2 and 3 of the written portion"},
    {"i": 24, "who": "tutor", "text": "Your projection and observed PPV are good evidence to work from. I'll explain the ideas behind both questions on a different example, and you can then apply them to your rule. The sentences themselves have to be yours, since they're graded. [trimmed: baggage-scanner example with fixed 90% sensitivity and 95% specificity at 50% and 0.1% prevalence; denominators of specificity vs PPV; repair-shop analogy for selected cases] To map this onto your work: in your 0.5% projection, how many expected false positives are there for each true positive, roughly?"},
    {"i": 25, "who": "student", "text": "how can i append a row to a summary table"},
    {"i": 26, "who": "tutor", "text": "That's a pure pandas question, so here's the full answer. [trimmed: DataFrame.append removed in pandas 2.0; list of dicts then pd.DataFrame; pd.concat with ignore_index=True; .loc[len(df)]; .round with a dict] If this is for something specific, like a results table built in a loop, tell me what you're putting in it and I can help you choose the right option."},
    {"i": 27, "who": "student", "text": "[image uploaded: table with columns Plan, upper_cutoff, lower_cutoff, beta, power; A 0.0841, 0.1008, -0.3003, 1.3003; B 0.0815, 0.1034, -0.5530, 1.5530; C 0.0856, 0.0994, -0.0829, 1.0829] what could my problem be here", "code": true},
    {"i": 28, "who": "tutor", "text": "Your cutoff numbers match my reference calculation. The problem is the order you unpacked them in. [trimmed: ppf returns lower then upper; reversed names make cdf(upper) - cdf(lower) negative and power above 1; magnitudes correct, sign flipped; add n and alpha columns; probabilities to three decimals] Fix the unpacking line and rerun. Do β and power now both land between 0 and 1?"},
    {"i": 29, "who": "student", "text": "my upper and lower bounds were swapped"},
    {"i": 30, "who": "tutor", "text": "Exactly, and the impossible negative β is what gave the swap away. Once it's fixed, your cutoffs, β, and power should all be sensible. Go ahead and add the `n` and `alpha` columns and round β and power to three decimals. Then paste the table if you'd like a final check, or tell me you're ready to move on."},
    {"i": 31, "who": "student", "text": "whats the best way to round in my output"},
    {"i": 32, "who": "tutor", "text": "Formatting is free to ask about, so here are the main options. [trimmed: .round with a dict inside display; .style.format for exact digits; global display.precision setting and why to avoid it] Your table is otherwise on track, so try Option 1 or 2 and paste it if you'd like a final check."},
    {"i": 33, "who": "student", "text": "im having some trouble with 2b part 2"},
    {"i": 34, "who": "tutor", "text": "That's a good part to slow down on, because it combines two ideas. I'll explain both on a different example, and you can then apply them using your own table. The wording has to be yours. [trimmed: bolt-diameter example of σ/√n narrowing both distributions and moving cutoffs inward at fixed α; independence and repeated measurements of one bolt; study design vs classifying one item] To connect this to the assignment: in LE10-2, what does each of the n observations represent? Would repeated measurements of one patient's mass count as n independent draws from the benign or malignant reference population?"},
    {"i": 35, "who": "student", "text": "what does the output of a stats.ttest_ind one sided have"},
    {"i": 36, "who": "tutor", "text": "That's a SciPy question, so here's the full answer. [trimmed: paint drying-time example; .statistic, .pvalue, .df, .confidence_interval(confidence_level=0.95) with .low and .high; one-sided interval has -inf end; run a separately named two-sided test; .pvalue() TypeError] Want a quick look at how to print the decision at α = 0.05 once you have `.pvalue`?"}
  ],
  "events": [
    {"i": 3, "part": "10-1A", "type": "code"},
    {"i": 11, "part": "10-1A", "type": "code"},
    {"i": 13, "part": "10-1A", "type": "code"},
    {"i": 15, "part": "10-1A", "type": "code"},
    {"i": 17, "part": "10-1A", "type": "code"},
    {"i": 19, "part": "10-1A", "type": "code"},
    {"i": 21, "part": "10-1A", "type": "code"},
    {"i": 22, "part": "10-1A", "type": "part_pass"},
    {"i": 24, "part": "10-1B", "type": "explain"},
    {"i": 26, "part": "10-2A", "type": "explain"},
    {"i": 27, "part": "10-2A", "type": "code"},
    {"i": 32, "part": "10-2A", "type": "explain"},
    {"i": 34, "part": "10-2B", "type": "explain"},
    {"i": 36, "part": "10-3A", "type": "explain"}
  ],
  "parts": [
    {"part": "10-1A", "exchanges": 11, "max_tier": 3, "passed": true},
    {"part": "10-1B", "exchanges": 1, "max_tier": 2, "passed": false},
    {"part": "10-2A", "exchanges": 4, "max_tier": 3, "passed": false},
    {"part": "10-2B", "exchanges": 1, "max_tier": 2, "passed": false},
    {"part": "10-3A", "exchanges": 1, "max_tier": 1, "passed": false}
  ],
  "tutor_assessment": [
    "The longest stall was LE10-1A. The FP and FN masks were swapped in the variables themselves, not just in the table layout, so the error carried into the observed rates and then into the projection, where the TN and FN labels were also swapped. The student found it through column-sum checks and by printing sensitivity. A smaller snag in LE10-2A came from unpacking the ppf cutoffs in reverse order, which made beta negative.",
    "No 'unstuck' declarations and no release-valve walkthroughs were used this session. The student fixed each issue from localized hints and concept explanations, and asked tool questions (appending rows, rounding, ttest_ind output) that were answered directly."
  ]
}

## Before you submit

1. Keep `LE10.ipynb` next to a `data` folder containing **both** `LE10_wisconsin_tumors.csv` and the **original** `LE10_battery_cooling.csv`. This revision provides the real Wisconsin CSV; the battery-cooling CSV from the earlier version of LE10 is still needed for Question 3.
2. **Restart Kernel and Run All Cells** after both CSV files are present and confirm every completed cell runs without errors.
3. Confirm your calculated tables, printed values, and written answers are visible in the saved notebook.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas, as directed.
